# 🚀 Amazing-QR: Tam Otomatik (CSV) QR Oluşturucu

Bu notebook, **Amazing-QR** projesini tamamen CSV (sipariş listesi) odaklı çalıştırmak için optimize edilmiştir. Manuel formlarla uğraşmanıza gerek yoktur.

---

## 🛠️ 1. Kurulum ve Hazırlık

Sistemi başlatmak için bu hücreyi çalıştırın.

In [ ]:
# Projeyi klonla
!git clone https://github.com/orioninsist/amazing-qr.git
%cd amazing-qr

# Gereksinimleri yükle
!pip install -r requirements.txt
!pip install . # amzqr paketini sisteme yükle

# Klasör yapısını hazırla
!mkdir -p inputs/assets
!mkdir -p output

## 📁 2. Sipariş ve Dosya Yükleme

`order.csv` dosyanızı ve varsa logolarınızı/GIF'lerinizi buraya yükleyin.
- `.csv` dosyaları otomatik olarak `inputs/` klasörüne taşınır.
- Görsel dosyaları otomatik olarak `inputs/assets/` klasörüne taşınır.

In [ ]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.csv'):
        shutil.move(fn, 'inputs/order.csv')
        print(f"✅ Sipariş dosyası güncellendi: inputs/order.csv")
    elif fn.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
        dest = f'inputs/assets/{fn}'
        shutil.move(fn, dest)
        print(f"🎨 Görsel/GIF yüklendi: {dest}")
    else:
        print(f"❓ Bilinmeyen dosya: {fn} (İşlenmedi)")

## 📊 3. Siparişleri Hazırla ve Önizle

Bu adım, eksik parametreleri (`level`, `version`, `colorized` vb.) otomatik olarak doldurur ve tüm listeyi (`words, version, level, picture, colorized, contrast, brightness, save_name`) gösterir.
- **Otomatik Dolum:** Sadece `words` ve `picture` girseniz bile sistem en iyi ayarları seçer.
- **Önizleme:** Listeyi kontrol edin, her şey hazırsa toplu işlemi başlatın.

In [ ]:
import pandas as pd
import os
import re

def slugify(value):
    value = re.sub(r'^https?://', '', str(value))
    value = re.sub(r'[^\w\s-]', '_', value).strip().lower()
    return re.sub(r'[-\s]+', '_', value)[:50]

csv_path = 'inputs/order.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    
    # --- Eksik Sütunları Akıllı Varsayılanlarla Doldur ---
    defaults = {
        'version': 1,
        'level': 'H',
        'contrast': 1.0,
        'brightness': 1.0,
        'colorized': None, # Dinamik belirlenecek
        'save_name': None  # Dinamik belirlenecek
    }
    
    for col, val in defaults.items():
        if col not in df.columns:
            df[col] = val
        else:
            df[col] = df[col].fillna(val)

    # Dinamik Varsayılanlar
    for i, row in df.iterrows():
        # 1. Colorized: Resim varsa True, yoksa False
        if pd.isna(row['colorized']) or str(row['colorized']).strip() == '':
            df.at[i, 'colorized'] = True if pd.notna(row.get('picture')) and str(row.get('picture')).strip() != '' and str(row.get('picture')).strip() != 'nan' else False
        
        # 2. Save Name: Otomatik isimlendirme
        if pd.isna(row['save_name']) or str(row['save_name']).strip() == '':
            ext = '.gif' if str(row.get('picture', '')).lower().endswith('.gif') else '.png'
            slug = slugify(row['words']) or f"qr_{i+1}"
            df.at[i, 'save_name'] = f"{slug}{ext}"

    # Sütun sırasını düzenle (İstendiği gibi 8 parametre)
    cols = ['words', 'version', 'level', 'picture', 'colorized', 'contrast', 'brightness', 'save_name']
    # Eksik kolonları ekle
    for c in cols:
        if c not in df.columns: df[c] = None
    
    df = df[cols]
    
    # Güncellenmiş listeyi kaydet (Batch işlem bunu kullanacak)
    df.to_csv(csv_path, index=False)
    
    print("✅ Sipariş listesi güncellendi ve hazır.")
    display(df)
else:
    print("⚠️ inputs/order.csv bulunamadı! Lütfen önce CSV dosyanızı yükleyin.")

## 🚀 4. Toplu İşlemi Başlat

Bu hücreyi çalıştırdığınızda tüm QR kodları otomatik olarak üretilecektir.

In [ ]:
!python batch_process.py

## 📥 5. Sonuçları İndir (ZIP)

Üretilen tüm QR kodlarını ve işlem raporunu ZIP olarak indirin.

In [ ]:
import shutil
from google.colab import files
import datetime

now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f'amazing_qr_results_{now}'

try:
    shutil.make_archive(zip_name, 'zip', 'output')
    files.download(f'{zip_name}.zip')
    print(f"✅ {zip_name}.zip indiriliyor...")
except Exception as e:
    print(f"❌ Hata: {e}")